# Capítulo 8 — Engenharia de dados para IA

## Missão
Montar um **pipeline de dados de treino e de avaliação** de qualidade, partindo de dados sujos. Ele vai:
1. inspecionar;
2. tirar duplicatas exatas e quase iguais (com MinHash feito à mão);
3. limpar e filtrar (inclusive dados pessoais);
4. medir a cobertura;
5. **gerar dados sintéticos** com o modelo onde faltam e verificar esses dados com um juiz;
6. formatar para treino.

Dados bons valem mais que um modelo maior.

**No fim deste caderno você terá:** o módulo `gabarito_assistente/dados_treino.py` (pipeline limpar → deduplicar → filtrar → formatar) e um conjunto final com relatório de cada etapa.

**Leitura essencial (opcional, livro em inglês):**
- Data Curation: Quality, Coverage, Quantity · PDF p. 389
- AI-Powered Data Synthesis · PDF p. 410
- Model Distillation · PDF p. 419
- Data Processing: Inspect, Deduplicate, Clean, Format · PDF p. 420

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_assistente/
os.makedirs("../gabarito_assistente", exist_ok=True)
open("../gabarito_assistente/__init__.py", "a").close()        # gabarito_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: gabarito_assistente")

## Ideia em 30 segundos
Três perguntas guiam a curadoria:
- **qualidade**: os exemplos estão certos?
- **cobertura**: eles cobrem os casos reais?
- **quantidade**: são suficientes?

Dados sintéticos (gerados por um modelo) tapam buracos de cobertura, mas precisam de verificação, porque o modelo também erra e repete padrões.

Dados sujos de propósito (já vem pronto): pares pergunta/resposta com duplicatas, quase duplicatas, vazios, inglês, textos longos demais e dados pessoais.

In [ ]:
sujos = [{"pergunta": p["pergunta"], "resposta": p["referencia"], "doc": p["doc"]} for p in PERGUNTAS]
sujos += [
    {"pergunta": "Quanto custa o plano Equipe?", "resposta": "R$ 29 por usuário por mês.", "doc": "planos"},
    {"pergunta": "quanto custa o plano equipe??", "resposta": "R$ 29 por usuário por mês.", "doc": "planos"},
    {"pergunta": "Quanto custa o plano Equipe por mês?", "resposta": "R$ 29 por usuário/mês.", "doc": "planos"},
    {"pergunta": "", "resposta": "Sem pergunta.", "doc": "planos"},
    {"pergunta": "How do I reset my password?", "resposta": "Click on forgot password.", "doc": "senha"},
    {"pergunta": "Meu e-mail é joana.silva@gmail.com e meu CPF 123.456.789-00, não consigo entrar.",
     "resposta": "Use Esqueci minha senha.", "doc": "senha"},
    {"pergunta": "Explique tudo sobre a Nimbus " + "por favor " * 80, "resposta": "...", "doc": "planos"},
]
df = pd.DataFrame(sujos)
print(len(df), "linhas")

### 8.1 **Inspeção.** Mostre: número de linhas, quantas perguntas vazias, a distribuição do tamanho das perguntas em palavras (`describe`) e as 3 maiores. Inspecionar antes de tudo evita surpresas.

In [ ]:
df["palavras"] = df.pergunta.str.split().str.len()
print("vazias:", (df.pergunta.str.strip() == "").sum())
display(df.palavras.describe())
df.nlargest(3, "palavras")[["pergunta", "palavras"]]

### 8.2 **Duplicatas exatas.** Normalize as perguntas (minúsculas, sem acento, sem pontuação, espaços únicos), calcule um hash (`hashlib.md5`) e remova as repetidas. Quantas saíram?

In [ ]:
import hashlib, string, unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode()
    return " ".join(t.translate(str.maketrans("", "", string.punctuation)).split())

df["hash"] = df.pergunta.map(lambda t: hashlib.md5(normalizar(t).encode()).hexdigest())
sem_exatas = df.drop_duplicates("hash")
print(len(df), "->", len(sem_exatas))

### 8.3 **Quase duplicatas com MinHash.** Implemente:
1. `shingles(t, k=3)`: o conjunto de trincas de palavras do texto normalizado;
2. a Jaccard exata;
3. um MinHash com 64 funções de hash (`hash((semente, s))`): a assinatura é o mínimo de cada função, e a estimativa da Jaccard é a fração de posições iguais.

Compare a Jaccard exata com a estimada para dois pares de perguntas e marque como quase duplicatas os pares com estimativa >= 0,5.

In [ ]:
def shingles(t, k=3):
    p = normalizar(t).split()
    return {" ".join(p[i:i + k]) for i in range(max(1, len(p) - k + 1))}

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

def minhash(conjunto, n=64):
    return np.array([min(hash((semente, s)) for s in conjunto) for semente in range(n)])

perguntas = sem_exatas.pergunta.tolist()
assinaturas = [minhash(shingles(p)) for p in perguntas]
pares = []
for i in range(len(perguntas)):
    for j in range(i + 1, len(perguntas)):
        estimada = float((assinaturas[i] == assinaturas[j]).mean())
        if estimada >= 0.5:
            pares.append((perguntas[i], perguntas[j], round(estimada, 2),
                          round(jaccard(shingles(perguntas[i]), shingles(perguntas[j])), 2)))
pd.DataFrame(pares, columns=["a", "b", "minhash", "jaccard exata"])

### 8.4 **Limpeza e filtros.** Escreva regras e aplique em `sem_exatas`:
- tirar perguntas vazias ou com mais de 60 palavras;
- tirar as que parecem inglês (heurística: palavras como `the`, `how`, `my`, `do`);
- **mascarar** e-mails e CPFs com regex, trocando por `[EMAIL]` e `[CPF]`.

Mostre quantas linhas sobraram e a pergunta mascarada.

In [ ]:
INGLES = {"the", "how", "my", "do", "i", "is", "what"}
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
CPF = re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b")

def parece_ingles(t):
    palavras = set(normalizar(t).split())
    return len(palavras & INGLES) >= 2

def mascarar(t):
    return CPF.sub("[CPF]", EMAIL.sub("[EMAIL]", t))

limpo = sem_exatas[(sem_exatas.pergunta.str.strip() != "") & (sem_exatas.palavras <= 60)]
limpo = limpo[~limpo.pergunta.map(parece_ingles)].copy()
limpo["pergunta"] = limpo.pergunta.map(mascarar)
print(len(sem_exatas), "->", len(limpo))
limpo[limpo.pergunta.str.contains(r"\[EMAIL\]|\[CPF\]")].pergunta.tolist()

### 8.5 **Cobertura.** Conte quantas perguntas há por documento (`doc`) e liste os documentos da Nimbus com **1 pergunta ou menos**. São esses os buracos que os dados sintéticos vão tapar.

In [ ]:
contagem = limpo.doc.value_counts()
poucas = [d["id"] for d in DOCUMENTOS if contagem.get(d["id"], 0) <= 1]
display(contagem.to_frame("perguntas"))
print("documentos com 1 pergunta ou menos:", poucas)

### 8.6 **Aumento tradicional (sem IA).** Gere variações de cada pergunta trocando palavras por sinônimos de um dicionário pequeno (por exemplo, `quanto custa` → `qual o preço de`, `consigo` → `é possível`). Quantas perguntas novas saíram?

In [ ]:
SINONIMOS = {"quanto custa": "qual o preço de", "consigo": "é possível", "posso": "é possível",
             "tem": "existe", "como": "de que jeito"}

def variacoes(t):
    saida = []
    for a, b in SINONIMOS.items():
        if a in t.lower():
            saida.append(re.sub(a, b, t, flags=re.I))
    return saida

novas = [(v, row.resposta, row.doc) for row in limpo.itertuples() for v in variacoes(row.pergunta)]
print(len(novas), "variações; exemplos:", [n[0] for n in novas[:3]])

### 8.7 **Dados sintéticos com IA.** Para cada documento com poucas perguntas, peça ao modelo 2 pares pergunta/resposta baseados **só** no texto do documento (JSON `{"itens": [{"pergunta": ..., "resposta": ...}]}`). Guarde em `sinteticos` com o `doc` de origem.

In [ ]:
sinteticos = []
for doc_id in poucas:
    doc = next(d for d in DOCUMENTOS if d["id"] == doc_id)
    j = perguntar_json("Crie 2 perguntas que um cliente faria e as respostas, usando SOMENTE este texto. "
                       'Devolva {"itens": [{"pergunta": "...", "resposta": "..."}]}\n\n' + doc["texto"],
                       temperature=0.7)
    for item in j.get("itens", [])[:2]:
        if isinstance(item, dict) and item.get("pergunta"):
            sinteticos.append({"pergunta": item["pergunta"], "resposta": item.get("resposta", ""), "doc": doc_id})
print(len(sinteticos), "pares sintéticos")
pd.DataFrame(sinteticos)

### 8.8 **Verificação com juiz.** Para cada par sintético, pergunte ao `MODELO_FORTE` se a resposta está correta e é sustentada pelo documento (JSON `{"relevante": true/false, "motivo": ...}`). Mantenha só os aprovados. Quantos passaram?

In [ ]:
aprovados = []
for s in sinteticos:
    doc = next(d["texto"] for d in DOCUMENTOS if d["id"] == s["doc"])
    v = perguntar_json(f"Documento: {doc}\nPergunta: {s['pergunta']}\nResposta: {s['resposta']}\n"
                       'A resposta está correta e sustentada pelo documento? Devolva {"relevante": true ou false, '
                       '"motivo": "..."}', modelo=MODELO_FORTE, temperature=0)
    if v.get("relevante") is True:
        aprovados.append(s)
print(f"{len(aprovados)}/{len(sinteticos)} aprovados")

### 8.9 **Módulo do pipeline.** Escreva `dados_treino.py` com `preparar_conjunto(pares, sistema)`, que recebe uma lista de dicts (`pergunta`, `resposta`) e faz, contando o que sai em cada etapa:
1. tira vazios;
2. tira duplicatas exatas;
3. tira perguntas com mais de 60 palavras;
4. mascara e-mail e CPF;
5. formata no padrão de conversa (`messages`).

Devolve `(exemplos, relatorio)`, em que `relatorio` é um dict etapa → quantidade.

_A célula grava `gabarito_assistente/dados_treino.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_assistente/dados_treino.py
"""Pipeline de preparação de dados de treino: limpar -> deduplicar -> filtrar -> formatar."""
import hashlib
import re
import string
import unicodedata

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
CPF = re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b")


def _normalizar(t: str) -> str:
    t = unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode()
    return " ".join(t.translate(str.maketrans("", "", string.punctuation)).split())


def _mascarar(t: str) -> str:
    return CPF.sub("[CPF]", EMAIL.sub("[EMAIL]", t))


def preparar_conjunto(pares: list[dict], sistema: str, max_palavras: int = 60):
    relatorio = {"entrada": len(pares)}
    pares = [p for p in pares if p.get("pergunta", "").strip() and p.get("resposta", "").strip()]
    relatorio["sem vazios"] = len(pares)
    vistos, unicos = set(), []
    for p in pares:
        h = hashlib.md5(_normalizar(p["pergunta"]).encode()).hexdigest()
        if h not in vistos:
            vistos.add(h)
            unicos.append(p)
    relatorio["sem duplicatas"] = len(unicos)
    curtos = [p for p in unicos if len(p["pergunta"].split()) <= max_palavras]
    relatorio["tamanho ok"] = len(curtos)
    exemplos = [{"messages": [{"role": "system", "content": sistema},
                              {"role": "user", "content": _mascarar(p["pergunta"])},
                              {"role": "assistant", "content": _mascarar(p["resposta"])}]} for p in curtos]
    relatorio["formatados"] = len(exemplos)
    return exemplos, relatorio


### 8.10 **Rode o pipeline completo.** Junte os dados sujos, as variações por sinônimo e os sintéticos aprovados, passe por `preparar_conjunto` e grave `../saidas/conjunto_final.jsonl`. Mostre o relatório como tabela.

In [ ]:
from gabarito_assistente.dados_treino import preparar_conjunto
todos = sujos + [{"pergunta": p, "resposta": r, "doc": d} for p, r, d in novas] + aprovados
exemplos, relatorio = preparar_conjunto(todos, sistema="Você é o suporte da Nimbus. Responda em até 3 frases.")
with open("../saidas/conjunto_final.jsonl", "w", encoding="utf-8") as f:
    for e in exemplos:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
pd.Series(relatorio, name="linhas").to_frame()

### 8.11 **Destilação.** Ideia: o modelo forte gera as respostas e um modelo menor aprende com elas. Para 3 perguntas dos `TICKETS`, gere a resposta de suporte com `MODELO_FORTE` ("professor") e monte os exemplos de treino do "aluno" no formato `messages`. Quanto isso custou (`gasto()`)?

In [ ]:
professor = []
for texto, _ in TICKETS[:3]:
    resposta = perguntar(texto, sistema="Você é o suporte da Nimbus. Responda em até 3 frases.", modelo=MODELO_FORTE)
    professor.append({"messages": [{"role": "user", "content": texto}, {"role": "assistant", "content": resposta}]})
print(json.dumps(professor[0], ensure_ascii=False, indent=1)[:400])
print(gasto())

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
